# Carnet de démarrage — sujet 07 · Santander Transaction

**Objectif du sujet** : détecter la transaction d'un client.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

10,05 % de positifs. Aucune variable ne porte de signal fort.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/1ftIdcRddFC_exiwx7i_yBKjEuXtnHdYz?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_07_Santander_Transaction/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |

**Format de sortie attendu** : `ID_code,target`.
**Colonne d'identifiant** : `ID_code`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


# Santander Transaction — Sujet 07

**Séance :** 01  
**Nom :** ZINSOUPLI Isidora  
**Porteur :** P3  

**Étapes assignées :**  
1.6 : trous  
1.4 : cardinalité  
1.9 : cibles

In [2]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = ": https://drive.google.com/drive/folders/1ftIdcRddFC_exiwx7i_yBKjEuXtnHdYz?usp=sharing"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [3]:
!find /kaggle/input -maxdepth 4

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/isidoradora
/kaggle/input/datasets/isidoradora/esgc-vak-sujet-07-santander
/kaggle/input/datasets/isidoradora/esgc-vak-sujet-07-santander/a_decider.csv
/kaggle/input/datasets/isidoradora/esgc-vak-sujet-07-santander/train.csv


In [4]:
import os
os.environ["DONNEES_PROJET"] = "/kaggle/input/datasets/isidoradora/esgc-vak-sujet-07-santander"

In [5]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_07_Santander_Transaction"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : /kaggle/input/datasets/isidoradora/esgc-vak-sujet-07-santander
fichiers vus : ['a_decider.csv', 'train.csv']


In [6]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()


In [7]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (160000, 202)
colonnes : 202


,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_68745,0,5.8891,-2.6946,8.5608,8.9821,9.5147,-11.7134,6.5543,16.4298,...,-3.4250,6.4394,0.0956,-1.9258,20.7324,-2.7677,2.7507,9.4133,13.8023,-20.7683
1,train_153215,0,9.8408,-3.7440,9.1857,10.0818,9.9471,-9.2351,5.7569,17.7236,...,10.1980,8.9795,1.6065,5.8926,15.7204,-2.1128,-7.1989,8.5131,16.7965,-0.8921
2,train_56308,0,9.1298,-9.5614,8.7906,2.7942,11.7535,-10.3463,6.1038,9.4427,...,5.3321,6.6036,0.9779,3.6963,21.5864,-1.6393,-5.3636,9.7069,14.3940,-5.7929
3,train_145961,0,6.8743,1.8831,10.9191,7.9922,10.1012,-20.3152,4.7830,18.6544,...,1.3300,3.3946,3.5116,-1.0296,10.9591,-0.3784,3.8469,8.9349,19.6068,6.3552
4,train_117483,0,8.5244,-4.8333,10.3487,5.2117,11.0081,-0.0894,5.3517,19.3375,...,-4.6720,5.0570,1.8477,3.3582,17.3930,-0.6751,-0.0803,11.1392,13.3615,-12.3896


In [8]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "target"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    160000.000000
mean          0.100487
std           0.300650
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: target, dtype: float64
target
0    0.899513
1    0.100487
Name: proportion, dtype: float64


In [9]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['ID_code', 'target']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"ID_code": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["target"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["ID_code"], complet=True)
identifiants = a_predire["ID_code"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


construction du cache Parquet : a_decider.parquet (une seule fois)
identifiants à prédire : 40000
écrit : soumission.csv · 40,000 lignes · colonnes ['ID_code', 'target']


,ID_code,target
0,train_69551,None
1,train_126730,None
2,train_133948,None
3,train_141297,None
4,train_101817,None
...,...,...
39995,train_126058,None
39996,train_136551,None
39997,train_10111,None
39998,train_147610,None


**Sujet 7: Santander Transaction**

In [10]:
#1.6 Trou

taux_manquant = entrainement.isna().mean() * 100

tableau_trous = pd.DataFrame({
    "Pourcentage_manquant": taux_manquant,
    "Pourcentage_rempli": 100 - taux_manquant
})

tableau_trous = tableau_trous.sort_values(by="Pourcentage_rempli", ascending=False)

display(tableau_trous)

,Pourcentage_manquant,Pourcentage_rempli
ID_code,0.0,100.0
target,0.0,100.0
var_0,0.0,100.0
var_1,0.0,100.0
var_2,0.0,100.0
...,...,...
var_195,0.0,100.0
var_196,0.0,100.0
var_197,0.0,100.0
var_198,0.0,100.0


In [11]:
#1.6 Trou - a_decider

a_decider = pd.read_csv(DOSSIER / "a_decider.csv")
print("a_decider :", a_decider.shape)

taux_manquant_ad = a_decider.isna().mean() * 100

tableau_trous_ad = pd.DataFrame({
    "Pourcentage_manquant": taux_manquant_ad,
    "Pourcentage_rempli": 100 - taux_manquant_ad
})

tableau_trous_ad = tableau_trous_ad.sort_values(by="Pourcentage_rempli", ascending=False)

print("Résultats pour a_decider :")
display(tableau_trous_ad)

a_decider : (40000, 201)
Résultats pour a_decider :


,Pourcentage_manquant,Pourcentage_rempli
ID_code,0.0,100.0
var_0,0.0,100.0
var_1,0.0,100.0
var_2,0.0,100.0
var_3,0.0,100.0
...,...,...
var_195,0.0,100.0
var_196,0.0,100.0
var_197,0.0,100.0
var_198,0.0,100.0


In [12]:
#1.6 Trou

manquants = entrainement.isna().sum()
print("Total de cases vides :", manquants.sum())
print("Colonnes avec au moins un trou :", (manquants > 0).sum())

Total de cases vides : 0
Colonnes avec au moins un trou : 0


In [13]:
#1.4 Cardinalité

valeurs_distinctes = entrainement.nunique()

tableau_cardinalite = pd.DataFrame({
    "Type": entrainement.dtypes.astype(str),
    "Valeurs_distinctes": valeurs_distinctes,
    "Pourcentage_des_lignes": (valeurs_distinctes / len(entrainement) * 100).round(2)
})

tableau_cardinalite = tableau_cardinalite.sort_values(by="Valeurs_distinctes")

print("Les 10 colonnes avec le moins de valeurs différentes :")
display(tableau_cardinalite.head(10))
print("Les 10 colonnes avec le plus de valeurs différentes :")
display(tableau_cardinalite.tail(10))

Les 10 colonnes avec le moins de valeurs différentes :


,Type,Valeurs_distinctes,Pourcentage_des_lignes
target,int64,2,0.00
var_68,float64,447,0.28
var_91,float64,7825,4.89
var_108,float64,8439,5.27
var_103,float64,9219,5.76
var_12,float64,9460,5.91
var_148,float64,10433,6.52
var_161,float64,10899,6.81
var_71,float64,13235,8.27
var_25,float64,14532,9.08


Les 10 colonnes avec le plus de valeurs différentes :


,Type,Valeurs_distinctes,Pourcentage_des_lignes
var_136,float64,131307,82.07
var_90,float64,131542,82.21
var_187,float64,131657,82.29
var_120,float64,132391,82.74
var_97,float64,132602,82.88
var_61,float64,133193,83.25
var_74,float64,134256,83.91
var_117,float64,136641,85.40
var_45,float64,140250,87.66
ID_code,object,160000,100.00


In [14]:
#Cible

effectifs = entrainement["target"].value_counts()
pourcentages = entrainement["target"].value_counts(normalize=True) * 100

tableau_cible = pd.DataFrame({
    "Nombre_de_clients": effectifs,
    "Pourcentage": pourcentages.round(2)
})
display(tableau_cible)

X = entrainement.drop(columns=["ID_code", "target"])
y = entrainement["target"]
print("X (les indices) :", X.shape)
print("y (la cible) :", y.shape)

,Nombre_de_clients,Pourcentage
target,,
0,143922,89.95
1,16078,10.05


X (les indices) : (160000, 200)
y (la cible) : (160000,)


In [15]:
train = pd.read_csv(DOSSIER / "train.csv")
print("train :", train.shape)

train : (160000, 202)


In [16]:
y = train["target"]
print(y.mean())

0.1004875


In [17]:
y = train["target"]
print(y.mean())

0.1004875


In [18]:
#2.3 Plancher : règle sans variable
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

# On met 25 % des clients de côté : c'est la partie de contrôle
X_app, X_val, y_app, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

# La règle : même note pour tous = part de 1 dans la partie apprentissage
valeur_regle = y_app.mean()
notes_controle = np.full(len(y_val), valeur_regle)

print("La règle : donner à chaque client la même note")
print("La valeur donnée :", round(valeur_regle, 4))
print("AUC sur la partie de contrôle :", roc_auc_score(y_val, notes_controle))

# Ce que cette règle décide : la note est sous 0.5, donc on ne contacte personne
decisions = (notes_controle >= 0.5).astype(int)
print("Exactitude de la règle (répondre toujours 0) :", round(accuracy_score(y_val, decisions), 4))

La règle : donner à chaque client la même note
La valeur donnée : 0.1005
AUC sur la partie de contrôle : 0.5
Exactitude de la règle (répondre toujours 0) : 0.8995


In [23]:
#2.7 Enregistrer l'essai
import pandas as pd
from datetime import datetime

PLANCHER = 0.5000            # règle sans variable, mesurée à l'étape 2.3
REFERENCE_ENONCE = 0.5639    # référence donnée par l'énoncé
MARGE = 0.01                 # écart minimal pour dire « battu » (choix à justifier)

essais = pd.DataFrame(columns=[
    "Essai", "Date", "Description", "Variables_utilisees",
    "Score_AUC", "Plancher_battu", "Bat_0.5639"])

def enregistrer_essai(description, variables, score):
    essais.loc[len(essais)] = [
        len(essais) + 1,
        datetime.now().strftime("%Y-%m-%d %H:%M"),
        description,
        variables,
        round(score, 4),
        "Oui" if score > PLANCHER + MARGE else "Non",
        "Oui" if score > REFERENCE_ENONCE else "Non",
    ]

enregistrer_essai("Même note pour tous", "aucune", 0.5000)
enregistrer_essai("Notes au hasard", "aucune", 0.5005)
enregistrer_essai("Meilleure variable seule", "1 variable (nom à noter)", 0.5631)
enregistrer_essai("Régression logistique", "200 variables (var_0 à var_199)", 0.8587)

display(essais)

,Essai,Date,Description,Variables_utilisees,Score_AUC,Plancher_battu,Bat_0.5639
0,1,2026-11-13 10:49,Même note pour tous,aucune,0.5000,Non,Non
1,2,2026-11-13 10:49,Notes au hasard,aucune,0.5005,Non,Non
2,3,2026-11-13 10:49,Meilleure variable seule,1 variable (nom à noter),0.5631,Oui,Non
3,4,2026-11-13 10:49,Régression logistique,200 variables (var_0 à var_199),0.8587,Oui,Oui
